# Homework: Operation Bell Drop — Entanglement-Based QKD
**Lecture 4.3 — Quantum Key Distribution**

---

**Briefing**

You are a quantum communications engineer at a secure messaging agency. Your mission: build an entanglement-based quantum key distribution system (the E91/BBM92 protocol), test it against an eavesdropper, and use it to encrypt a classified message with a one-time pad.

In BB84 (which we studied in lecture), Alice *prepares* qubits and sends them to Bob. In this homework you'll implement the **entanglement-based** variant: a source produces Bell pairs $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$, sending one qubit to Alice and one to Bob. They each independently choose a random measurement basis ($Z$ or $X$), measure, and later compare bases over a public channel.

---

**Instructions**

- Complete every cell marked `### YOUR CODE HERE ###`.
- Do **not** rename variables that appear in the prompts — those names may be checked by the grader.
- Short-answer cells are graded on correctness and clarity, not length.
- Before submitting: `Kernel → Restart & Run All` and confirm zero errors.
- Upload this `.ipynb` file to Gradescope.

---

| Part | Topic | Points |
|------|-------|--------|
| A | Bell pairs and random measurement | 20 |
| B | Sifting and key extraction | 25 |
| C | Eavesdropper detection | 30 |
| D | One-time pad encryption | 25 |
| **Total** | | **100** |

In [ ]:
# Run this cell first — standard imports for the entire assignment
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
import numpy as np
import matplotlib.pyplot as plt

simulator = AerSimulator()
rng = np.random.default_rng(seed=42)  # reproducible randomness

---
## Part A — Bell Pairs and Random Measurement (20 pts)

### Background

In the entanglement-based QKD protocol, a source distributes Bell pairs $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$. Alice receives qubit 0 and Bob receives qubit 1.

Each party independently and randomly chooses to measure in either the **$Z$-basis** ($\{|0\rangle, |1\rangle\}$) or the **$X$-basis** ($\{|+\rangle, |-\rangle\}$). Measuring in the $X$-basis is implemented by applying a Hadamard gate before a $Z$-measurement.

**Key physics:** When Alice and Bob both measure $|\Phi^+\rangle$ in the *same* basis, their outcomes are **perfectly correlated**: both get 0 or both get 1, with equal probability. When they measure in *different* bases, their outcomes are completely uncorrelated.

### A1 (5 pts) — Verify Bell pair correlations with Qiskit

Build four 2-qubit circuits, each starting from $|\Phi^+\rangle$, for the four basis combinations: (Z,Z), (Z,X), (X,Z), (X,X). Measure each for 4096 shots.

To measure in the $X$-basis, apply `qc.h(qubit)` before `qc.measure(qubit, cbit)`.

Store your results in a dictionary `correlation_counts` with keys `'ZZ'`, `'ZX'`, `'XZ'`, `'XX'` and values being the counts dictionaries from the simulator.

Print the counts for each basis combination. You should see:
- **Same basis (ZZ, XX):** only `'00'` and `'11'` appear (perfect correlation)
- **Different basis (ZX, XZ):** all four outcomes `'00'`, `'01'`, `'10'`, `'11'` appear roughly equally

In [ ]:
### YOUR CODE HERE ###

basis_combos = ['ZZ', 'ZX', 'XZ', 'XX']
correlation_counts = {}

for bases in basis_combos:
    qc = QuantumCircuit(2, 2)
    
    # Step 1: Create |Phi+>
    # ...
    
    # Step 2: Apply H before measurement if measuring in X basis
    # Alice's basis = bases[0], Bob's basis = bases[1]
    # ...
    
    # Step 3: Measure
    qc.measure([0, 1], [0, 1])
    
    counts = simulator.run(qc, shots=4096).result().get_counts()
    correlation_counts[bases] = counts

for bases, counts in correlation_counts.items():
    print(f"Bases {bases}: {counts}")

### A2 (5 pts) — Short answer

In 3–4 sentences, explain:
1. Why do Alice and Bob get perfectly correlated results when they choose the same basis?
2. Why are their results uncorrelated when they choose different bases?

*Hint:* Think about what $|\Phi^+\rangle$ looks like when expanded in the $X$-basis.

**Your answer:**

*(double-click to edit)*

### A3 (10 pts) — Simulate many Bell pair measurements

Now scale up. Write a function `measure_bell_pairs(n_pairs, rng)` that simulates `n_pairs` Bell pair measurements **classically** (using numpy, not Qiskit circuits — this is much faster for large numbers of pairs).

The simulation should:
1. For each pair, randomly choose Alice's basis and Bob's basis (each independently $Z$ or $X$ with equal probability).
2. Simulate the measurement outcomes using the physics:
   - **Same basis:** Both get the same random bit (0 or 1 with equal probability).
   - **Different basis:** Each gets an independent random bit.
3. Return four numpy arrays: `alice_bases`, `bob_bases`, `alice_bits`, `bob_bits`, each of length `n_pairs`. Use `0` for $Z$ and `1` for $X$.

Run with `n_pairs = 10000` and verify:
- About 50% of the time, Alice and Bob chose the same basis.
- When they chose the same basis, their bits agree 100% of the time.

In [ ]:
def measure_bell_pairs(n_pairs, rng):
    """
    Simulate n_pairs Bell pair measurements.
    
    Returns:
        alice_bases: array of 0 (Z) or 1 (X), length n_pairs
        bob_bases:   array of 0 (Z) or 1 (X), length n_pairs
        alice_bits:  array of 0 or 1, length n_pairs
        bob_bits:    array of 0 or 1, length n_pairs
    """
    ### YOUR CODE HERE ###
    
    # 1. Random basis choices
    alice_bases = rng.integers(0, 2, size=n_pairs)
    bob_bases   = rng.integers(0, 2, size=n_pairs)
    
    # 2. Alice's measurement outcomes (always random)
    alice_bits = rng.integers(0, 2, size=n_pairs)
    
    # 3. Bob's outcomes depend on whether bases match
    bob_bits = np.zeros(n_pairs, dtype=int)
    same_basis = ...   # boolean mask where alice_bases == bob_bases
    diff_basis = ...   # boolean mask where alice_bases != bob_bases
    
    # Same basis → perfectly correlated (Bob = Alice)
    # ...
    
    # Different basis → independent random bit
    # ...
    
    return alice_bases, bob_bases, alice_bits, bob_bits


# Test it
n_pairs = 10000
test_rng = np.random.default_rng(seed=123)
a_bases, b_bases, a_bits, b_bits = measure_bell_pairs(n_pairs, test_rng)

same = (a_bases == b_bases)
frac_same = np.mean(same)
agree_when_same = np.mean(a_bits[same] == b_bits[same])

print(f"Fraction with same basis: {frac_same:.3f}  (expected ~0.500)")
print(f"Agreement when same basis: {agree_when_same:.4f}  (expected 1.0000)")

---
## Part B — Sifting and Key Extraction (25 pts)

### Background

After measuring all their Bell pairs, Alice and Bob publicly announce their **basis choices** (but not their measurement results). They **sift** their data: they keep only the rounds where they chose the same basis, and discard the rest. The surviving bits form the **raw key**.

In a perfect (noiseless, no eavesdropper) scenario, the sifted bits should agree perfectly between Alice and Bob.

### B1 (10 pts) — Implement sifting

Write a function `sift_keys(alice_bases, bob_bases, alice_bits, bob_bits)` that:
1. Finds the indices where `alice_bases == bob_bases`.
2. Extracts Alice's and Bob's bits at those indices.
3. Returns `alice_key, bob_key` (both 1D numpy arrays).

Run it on `n_pairs = 10000` Bell pairs and print:
- The number of sifted bits (should be ~5000).
- The first 20 bits of each key (should match exactly).
- The **quantum bit error rate (QBER)**: the fraction of sifted positions where Alice and Bob disagree. It should be 0.0 with no eavesdropper.

In [ ]:
def sift_keys(alice_bases, bob_bases, alice_bits, bob_bits):
    """
    Keep only the bits where Alice and Bob chose the same basis.
    
    Returns:
        alice_key: 1D numpy array of Alice's sifted bits
        bob_key:   1D numpy array of Bob's sifted bits
    """
    ### YOUR CODE HERE ###
    pass


def compute_qber(alice_key, bob_key):
    """
    Compute the quantum bit error rate: fraction of positions where keys disagree.
    """
    ### YOUR CODE HERE ###
    pass


# Generate data and sift
n_pairs = 10000
rng_b = np.random.default_rng(seed=42)
a_bases, b_bases, a_bits, b_bits = measure_bell_pairs(n_pairs, rng_b)
alice_key, bob_key = sift_keys(a_bases, b_bases, a_bits, b_bits)

print(f"Total pairs:  {n_pairs}")
print(f"Sifted bits:  {len(alice_key)}  (expected ~{n_pairs//2})")
print(f"\nAlice's first 20 bits: {alice_key[:20]}")
print(f"Bob's   first 20 bits: {bob_key[:20]}")
print(f"\nQBER = {compute_qber(alice_key, bob_key):.4f}  (expected 0.0000)")

### B2 (10 pts) — Verify with Qiskit (small scale)

As a cross-check, repeat the protocol using actual Qiskit circuits for a small number of pairs (`n_pairs_qiskit = 100`). For each pair:

1. Build a 2-qubit circuit that creates $|\Phi^+\rangle$.
2. Apply $H$ to qubit 0 if Alice chose the $X$ basis. Apply $H$ to qubit 1 if Bob chose the $X$ basis.
3. Measure both qubits and run for 1 shot.
4. Extract Alice's bit (qubit 0) and Bob's bit (qubit 1) from the outcome string.

Sift and compute QBER. It should again be 0.0.

*Note:* This is slow because we run one circuit per pair. That's why we use the classical simulation for the rest of the homework — but this confirms the Qiskit physics matches.

In [ ]:
### YOUR CODE HERE ###

n_pairs_qiskit = 100
rng_q = np.random.default_rng(seed=99)

alice_bases_q = rng_q.integers(0, 2, size=n_pairs_qiskit)
bob_bases_q   = rng_q.integers(0, 2, size=n_pairs_qiskit)
alice_bits_q  = np.zeros(n_pairs_qiskit, dtype=int)
bob_bits_q    = np.zeros(n_pairs_qiskit, dtype=int)

for i in range(n_pairs_qiskit):
    qc = QuantumCircuit(2, 2)
    
    # Create |Phi+>
    # ...
    
    # Apply H if measuring in X basis
    # ...
    
    # Measure
    qc.measure([0, 1], [0, 1])
    
    result = simulator.run(qc, shots=1).result().get_counts()
    outcome = list(result.keys())[0]  # e.g. '01'
    
    # Qiskit ordering: outcome = 'q1 q0', so outcome[1] = qubit 0 (Alice)
    alice_bits_q[i] = int(outcome[1])
    bob_bits_q[i]   = int(outcome[0])

alice_key_q, bob_key_q = sift_keys(alice_bases_q, bob_bases_q, alice_bits_q, bob_bits_q)
qber_qiskit = compute_qber(alice_key_q, bob_key_q)

print(f"Qiskit verification ({n_pairs_qiskit} pairs):")
print(f"  Sifted bits: {len(alice_key_q)}")
print(f"  QBER = {qber_qiskit:.4f}  (expected 0.0000)")

### B3 (5 pts) — Short answer

1. Why do Alice and Bob discard rounds where they chose different bases? What would happen to the QBER if they kept those rounds?
2. In BB84, Alice *chooses* to prepare in either the $Z$ or $X$ basis. In the Bell-based protocol, Alice always measures half of $|\Phi^+\rangle$. Explain in 2–3 sentences why the sifting step works the same way in both protocols.

**Your answer:**

*(double-click to edit)*

---
## Part C — Eavesdropper Detection (30 pts)

### Background

An eavesdropper (Eve) intercepts Bob's qubit from the Bell pair, measures it in a randomly chosen basis ($Z$ or $X$), and sends Bob a fresh qubit prepared in the state she measured. This is called an **intercept-resend** attack.

Eve's attack disrupts the entanglement. When Eve measures in a different basis than Alice and Bob, she introduces errors. The expected QBER under full interception is:

$$\text{QBER}_{\text{Eve}} = \frac{1}{2} \times 0 + \frac{1}{2} \times \frac{1}{2} = 25\%$$

The first factor: Eve picks the same basis as Alice and Bob 50% of the time (no error). The second: when Eve picks the wrong basis, Bob gets a random result, disagreeing with Alice 50% of the time.

### C1 (15 pts) — Simulate Eve's intercept-resend attack

Write a function `measure_bell_pairs_with_eve(n_pairs, p_intercept, rng)` that modifies your `measure_bell_pairs` function to include Eve.

For each Bell pair:
1. With probability `p_intercept`, Eve intercepts Bob's qubit:
   - Eve chooses a random basis ($Z$ or $X$).
   - Eve measures Bob's qubit. Since the Bell pair collapses, Eve's result is a random bit. **Alice's bit is determined by the Bell state physics:** if Eve's basis matches Alice's, Alice's bit equals Eve's bit (correlation preserved through the entanglement). If Eve's basis differs from Alice's, Alice gets an independent random bit.
   - Eve sends Bob a qubit in the state she measured. Now Bob measures it in his basis: if Bob's basis matches Eve's, Bob gets Eve's bit. If not, Bob gets a random bit.
2. With probability `1 - p_intercept`, no interception — same as before (perfect correlations when bases match).

Return: `alice_bases, bob_bases, alice_bits, bob_bits` (same format as before).

**Test your function:**
- `p_intercept = 0.0` → QBER ≈ 0%
- `p_intercept = 1.0` → QBER ≈ 25%
- `p_intercept = 0.5` → QBER ≈ 12.5%

In [ ]:
def measure_bell_pairs_with_eve(n_pairs, p_intercept, rng):
    """
    Simulate Bell pair measurements with Eve's intercept-resend attack.
    
    Args:
        n_pairs: number of Bell pairs
        p_intercept: probability Eve intercepts each pair (0 to 1)
        rng: numpy random generator
    
    Returns:
        alice_bases, bob_bases, alice_bits, bob_bits (numpy arrays)
    """
    ### YOUR CODE HERE ###
    
    alice_bases = rng.integers(0, 2, size=n_pairs)
    bob_bases   = rng.integers(0, 2, size=n_pairs)
    alice_bits  = np.zeros(n_pairs, dtype=int)
    bob_bits    = np.zeros(n_pairs, dtype=int)
    
    # Which pairs does Eve intercept?
    eve_intercepts = rng.random(n_pairs) < p_intercept
    eve_bases      = rng.integers(0, 2, size=n_pairs)
    
    # ---- No interception: same as measure_bell_pairs ----
    no_eve = ~eve_intercepts
    # Alice gets a random bit
    # Bob: same basis → copy Alice's bit; different basis → independent random
    # ...
    
    # ---- Eve intercepts ----
    # Eve measures Bob's qubit → gets a random bit (eve_bit)
    # Alice's bit depends on whether her basis matches Eve's basis:
    #   match → Alice gets same bit as Eve (entanglement correlation)
    #   mismatch → Alice gets independent random bit
    # Bob receives Eve's re-prepared qubit. His bit depends on whether
    # his basis matches Eve's basis:
    #   match → Bob gets Eve's bit
    #   mismatch → Bob gets independent random bit
    # ...
    
    return alice_bases, bob_bases, alice_bits, bob_bits


# Test
n_test = 20000
for p in [0.0, 0.5, 1.0]:
    rng_test = np.random.default_rng(seed=42)
    ab, bb, abits, bbits = measure_bell_pairs_with_eve(n_test, p, rng_test)
    ak, bk = sift_keys(ab, bb, abits, bbits)
    qber = compute_qber(ak, bk)
    expected = p * 0.25
    print(f"p_intercept = {p:.1f}  →  QBER = {qber:.4f}  (expected ~{expected:.4f})")

### C2 (10 pts) — QBER vs. interception fraction

Sweep `p_intercept` from 0 to 1 in steps of 0.05 and compute the QBER for each value. Use `n_pairs = 20000` for each point.

Plot QBER (y-axis) vs. `p_intercept` (x-axis). Also plot the theoretical prediction $\text{QBER} = 0.25 \times p$ as a dashed line.

Store your arrays as `p_values` and `qber_values`.

Add a horizontal dashed line at QBER = 0.11 (the typical security threshold). Label it "Security threshold".

In [ ]:
### YOUR CODE HERE ###

p_values = np.arange(0, 1.05, 0.05)
qber_values = np.zeros(len(p_values))

for i, p in enumerate(p_values):
    rng_sweep = np.random.default_rng(seed=i)
    # ... simulate, sift, compute QBER ...
    pass

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
# ... your plot code here ...
# Include: data points, theoretical line, security threshold, axis labels, legend
plt.show()

### C3 (5 pts) — Short answer

1. Derive the 25% QBER formula for full interception ($p=1$). Walk through the probability tree: what fraction of the time does Eve choose the wrong basis, and when she does, what fraction of Bob's sifted bits are flipped? (3–4 sentences)

2. If Alice and Bob observe a QBER of 15%, what can they conclude? Can they determine the exact value of $p_{\text{intercept}}$? (2–3 sentences)

**Your answer:**

*(double-click to edit)*

---
## Part D — One-Time Pad Encryption (25 pts)

### Background

The whole point of QKD is to generate a shared secret key for encryption. The **one-time pad** (OTP) is the only encryption scheme that is provably unbreakable — but it requires a key that is (1) as long as the message, (2) truly random, and (3) used only once. QKD provides exactly that.

The one-time pad works by XOR-ing the plaintext with the key:

$$\text{ciphertext} = \text{plaintext} \oplus \text{key}$$
$$\text{plaintext} = \text{ciphertext} \oplus \text{key}$$

We'll convert text to binary using 8-bit ASCII encoding.

### D1 (5 pts) — Text-to-binary conversion

Complete the helper functions below. `text_to_bits` converts a string to a numpy array of 0s and 1s (8 bits per character, using ASCII). `bits_to_text` converts back.

Test by encoding `"QUANTUM"` to bits and back.

In [ ]:
def text_to_bits(text):
    """
    Convert a text string to a numpy array of bits (8 bits per character).
    Example: 'A' → [0,1,0,0,0,0,0,1] (ASCII 65 = 01000001)
    """
    ### YOUR CODE HERE ###
    pass


def bits_to_text(bits):
    """
    Convert a numpy array of bits back to a text string.
    """
    ### YOUR CODE HERE ###
    pass


# Test
test_msg = "QUANTUM"
test_bits = text_to_bits(test_msg)
print(f"'{test_msg}' → {test_bits}")
print(f"Length: {len(test_bits)} bits ({len(test_msg)} characters × 8 bits)")
print(f"Back to text: '{bits_to_text(test_bits)}'")

### D2 (10 pts) — Encrypt and decrypt a classified message

Now put the full protocol together:

1. Alice and Bob share Bell pairs and sift keys (no Eve). Use enough pairs so the sifted key is at least as long as the message.
2. Alice encrypts the message `"MEET AT DAWN"` using the one-time pad (XOR with key).
3. Alice sends the ciphertext to Bob over a public classical channel.
4. Bob decrypts using his copy of the key.

Store the encrypted bits in `ciphertext` and the decrypted string in `decrypted_message`.

Print the plaintext, ciphertext (as a bit string), and decrypted message.

In [ ]:
### YOUR CODE HERE ###

secret_message = "MEET AT DAWN"
msg_bits = text_to_bits(secret_message)
n_bits_needed = len(msg_bits)

# Generate enough Bell pairs (need ~2x bits because of sifting)
n_pairs_otp = n_bits_needed * 3  # safety margin
rng_otp = np.random.default_rng(seed=2026)

# Run protocol (no Eve)
ab, bb, abits, bbits = measure_bell_pairs(n_pairs_otp, rng_otp)
alice_key_otp, bob_key_otp = sift_keys(ab, bb, abits, bbits)

# Trim keys to message length
alice_key_otp = alice_key_otp[:n_bits_needed]
bob_key_otp   = bob_key_otp[:n_bits_needed]

# Encrypt (Alice)
ciphertext = ...  # XOR msg_bits with alice_key_otp

# Decrypt (Bob)
decrypted_bits = ...  # XOR ciphertext with bob_key_otp
decrypted_message = bits_to_text(decrypted_bits)

print(f"Plaintext:  '{secret_message}'")
print(f"Ciphertext: {''.join(map(str, ciphertext))}")
print(f"Decrypted:  '{decrypted_message}'")
print(f"\nSuccess: {decrypted_message == secret_message}")

### D3 (10 pts) — Eve corrupts the message

Now repeat the protocol with Eve intercepting 100% of the pairs (`p_intercept = 1.0`). Alice and Bob **don't check for Eve** — they just sift and use the key.

1. Generate keys with full Eve interception.
2. Alice encrypts the same message using her key.
3. Bob decrypts using his (now different) key.
4. Print the garbled decrypted text.

Then answer: What fraction of the decrypted bits are wrong? Does this match the QBER?

Store Bob's garbled decryption in `garbled_message`.

In [ ]:
### YOUR CODE HERE ###

# Same message
secret_message_eve = "MEET AT DAWN"
msg_bits_eve = text_to_bits(secret_message_eve)
n_bits_eve = len(msg_bits_eve)

# Generate keys with Eve (p=1.0)
n_pairs_eve = n_bits_eve * 3
rng_eve = np.random.default_rng(seed=2026)

ab_e, bb_e, abits_e, bbits_e = measure_bell_pairs_with_eve(n_pairs_eve, 1.0, rng_eve)
alice_key_eve, bob_key_eve = sift_keys(ab_e, bb_e, abits_e, bbits_e)

alice_key_eve = alice_key_eve[:n_bits_eve]
bob_key_eve   = bob_key_eve[:n_bits_eve]

# Encrypt with Alice's key
ciphertext_eve = ...  # XOR msg_bits_eve with alice_key_eve

# Bob decrypts with his (corrupted) key
garbled_bits = ...  # XOR ciphertext_eve with bob_key_eve
garbled_message = bits_to_text(garbled_bits)

# Bit error rate in the decrypted message
bit_errors = np.mean(msg_bits_eve != garbled_bits)

print(f"Plaintext:         '{secret_message_eve}'")
print(f"Bob decrypts:      '{garbled_message}'")
print(f"Bit error rate:    {bit_errors:.2%}")
print(f"Key disagreement:  {np.mean(alice_key_eve != bob_key_eve):.2%}")
print(f"\nEve has broken the secure channel — the message is garbled!")
print(f"This is why Alice and Bob must check QBER before using the key.")

### D4 (5 pts, bonus) — The full protocol with security check

Put together the complete secure protocol:

1. Alice and Bob generate keys using Bell pairs (with some `p_intercept`).
2. They sacrifice a random subset (say, 20%) of their sifted key to estimate the QBER.
3. If QBER > 11%, they abort and print a warning.
4. If QBER ≤ 11%, they use the remaining 80% of the key to encrypt and decrypt the message.

Test with `p_intercept = 0.0` (should succeed) and `p_intercept = 0.8` (should abort).

*This problem is bonus — no points deducted if skipped.*

In [ ]:
def secure_qkd_protocol(message, p_intercept, n_pairs=10000, security_threshold=0.11, seed=42):
    """
    Full QKD protocol with security check.
    
    Returns:
        success: True if key was secure and message transmitted
        result: decrypted message (if success) or warning string (if abort)
    """
    ### YOUR CODE HERE ###
    pass


# Test: No Eve → should succeed
print("=" * 50)
print("Test 1: No eavesdropper")
print("=" * 50)
success, result = secure_qkd_protocol("MEET AT DAWN", p_intercept=0.0)
print(f"Success: {success}")
print(f"Result:  {result}")

print()

# Test: Heavy Eve → should abort
print("=" * 50)
print("Test 2: Eve intercepts 80%")
print("=" * 50)
success, result = secure_qkd_protocol("MEET AT DAWN", p_intercept=0.8)
print(f"Success: {success}")
print(f"Result:  {result}")

---

**Mission debrief:** You've built a complete entanglement-based QKD system — from Bell pair distribution, through sifting and eavesdropper detection, to secure encryption. The security of this system doesn't rest on computational assumptions (like RSA) but on the laws of quantum mechanics: the no-cloning theorem ensures Eve can't copy qubits, and entanglement ensures any interception disturbs the correlations in a detectable way.

---

**Before submitting:** `Kernel → Restart & Run All` and confirm zero errors. Upload this `.ipynb` to Gradescope.